In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm

%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

from IPython.display import Image as ShowImage, display
from PIL import Image

def save_gif(frames, name, duration=80):
    # write an animation (list of uint8 RGB frames) as a GIF and show it
    path = f'figures/{name}.gif' if SAVE_FIGURES else f'{name}.gif'
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    gif = [Image.fromarray(f) for f in frames]
    palette = gif[0].quantize(192)                       # one palette, no dithering: a small file
    gif = [g.quantize(palette=palette, dither=Image.Dither.NONE) for g in gif]
    gif[0].save(path, save_all=True, append_images=gif[1:], duration=duration, loop=0, optimize=True)
    display(ShowImage(filename=path))

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image_cc_1.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_cc_1.jpg
wget --no-verbose --output-document=image_cc_2.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_cc_2.jpg

mkdir -p 15
wget --no-verbose --output-document=15/texture_facade1.jpg https://github.com/chrirupp/cv_course/raw/main/data/texture_facade1.jpg
wget --no-verbose --output-document=15/texture_facade2.jpg https://github.com/chrirupp/cv_course/raw/main/data/texture_facade2.jpg
wget --no-verbose --output-document=15/texture_facade3.jpg https://github.com/chrirupp/cv_course/raw/main/data/texture_facade3.jpg
wget --no-verbose --output-document=15/texture_road.jpg https://github.com/chrirupp/cv_course/raw/main/data/texture_road.jpg
wget --no-verbose --output-document=15/texture_paving.jpg https://github.com/chrirupp/cv_course/raw/main/data/texture_paving.jpg
wget --no-verbose --output-document=15/texture_bricks.jpg https://github.com/chrirupp/cv_course/raw/main/data/texture_bricks.jpg

# Lecture 15 - Multiple View Geometry

Two parts:

* **A rendered street** (the ray caster of Lecture 14): the cameras are known exactly, so we can draw the true epipolar lines and epipoles, check estimates against the true fundamental matrix, and compare triangulated and stereo depth with the truth.
* **Two real photos** of Christ Church Meadow Building: the fundamental matrix with the eight-point algorithm (and why normalisation and the rank-2 constraint matter), what goes wrong when the scene is (almost) a plane, and a two-view reconstruction by triangulation.

The last section is about stereo: rectification, disparity and how accurate depth from two views can be.

No GPU needed. Rendering takes a few minutes.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

## A rendered street: two cameras we know exactly

The small ray caster of Lecture 14 (copied here): a camera $K [R \mid \mathbf{t}]$ sends one ray per pixel into a street of textured boxes. `photo(..., depth=True)` also returns the depth $Z$ of every pixel, which gives us the true correspondences between two views.

A reminder of the convention: the world's $y$ axis points up, the camera's $x$ right, $y$ down and $z$ forward. With `look_at`, world $+x$ is therefore to the camera's *left*.

In [ ]:
def intrinsics(f, cx=320, cy=240):
    return np.array([[f, 0, cx], [0, f, cy], [0, 0, 1.0]])

def look_at(camera_center, target, up=(0, 1, 0)):
    """rotation R and translation t of a camera at `camera_center` looking at `target` (x right, y down, z forward)"""
    z = np.asarray(target, float) - camera_center
    z /= np.linalg.norm(z)
    x = np.cross(z, up)
    x /= np.linalg.norm(x)
    y = np.cross(z, x)
    R = np.stack([x, y, z])
    return R, -R @ camera_center

def project(X, K, R, t):
    """perspective projection of (N, 3) world points to (N, 2) pixels"""
    x = (K @ (R @ X.T + t[:, None])).T
    return x[:, :2] / x[:, 2:]

def project_orthographic(X, K, R, t, scale):
    """orthographic projection: drop the depth instead of dividing by it"""
    Xc = (R @ X.T + t[:, None]).T
    return Xc[:, :2] * scale + K[:2, 2]


def camera_rays(K, R, t, width, height, k1=0.0, ortho=None):
    # one ray per pixel: origins and unit directions in world coordinates, shape (H, W, 3);
    # k1: radial distortion; ortho: pixels per metre of an orthographic camera (parallel rays)
    u, v = np.meshgrid(np.arange(width) + 0.5, np.arange(height) + 0.5)
    xn = (u - K[0, 2]) / K[0, 0]                     # K^-1 x: normalised image coordinates
    yn = (v - K[1, 2]) / K[1, 1]
    valid = np.ones_like(xn, bool)
    if k1:                                           # undo x_d = x (1 + k1 r^2) by fixed-point iteration
        xu, yu = xn.copy(), yn.copy()
        for _ in range(50):
            s = 1 + k1 * (xu ** 2 + yu ** 2)
            xu, yu = xn / s, yn / s
        s = 1 + k1 * (xu ** 2 + yu ** 2)
        valid = (np.abs(xu * s - xn) + np.abs(yu * s - yn) < 1e-4) & (s > 0)   # pixels no ray maps to stay black
        xn, yn = xu, yu
    C = -R.T @ t                                     # camera centre in world coordinates
    if ortho:
        offsets = np.stack([(u - K[0, 2]) / ortho, (v - K[1, 2]) / ortho, np.zeros_like(u)], -1)
        origins = C + offsets @ R - 1000 * R[2]      # rays start on a plane (far behind C, so nothing is cut off) ...
        dirs = np.broadcast_to(R[2], origins.shape).copy()   # ... and all point along the optical axis
    else:
        dirs = np.stack([xn, yn, np.ones_like(xn)], -1) @ R   # R^T (x_n, y_n, 1): through the camera centre
        dirs /= np.linalg.norm(dirs, axis=-1, keepdims=True)
        origins = np.broadcast_to(C, dirs.shape).copy()
    return origins, dirs, valid

LIGHT = np.array([0.4, 1.0, -0.6]) / np.linalg.norm([0.4, 1.0, -0.6])

def texture(name, width, height):
    # a photo texture (ambientCG, CC0) that repeats every width x height metres; s, t in metres (t up).
    # Far away (or seen at a grazing angle) one pixel covers many texels: reading a single texel would alias
    # (shimmering, moire). Like a GPU we keep a "mipmap", the photo at half, quarter, ... resolution, and read
    # the level whose texels are about as large as the pixel's footprint (bilinear, blended between two levels).
    levels = [plt.imread(f'15/texture_{name}.jpg') / 255.0]
    while min(levels[-1].shape[:2]) > 4:
        levels.append(cv2.resize(levels[-1], None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA))
    texels_per_metre = max(levels[0].shape[1] / width, levels[0].shape[0] / height)
    def bilinear(image, s, t):
        h, w = image.shape[:2]
        x, y = np.mod(s / width, 1) * w - 0.5, (1 - np.mod(t / height, 1)) * h - 0.5
        x0, y0 = np.floor(x).astype(int), np.floor(y).astype(int)
        fx, fy = (x - x0)[:, None], (y - y0)[:, None]
        x0, x1, y0, y1 = x0 % w, (x0 + 1) % w, y0 % h, (y0 + 1) % h
        return (1 - fy) * ((1 - fx) * image[y0, x0] + fx * image[y0, x1]) + fy * ((1 - fx) * image[y1, x0] + fx * image[y1, x1])
    def lookup(s, t, footprint):
        # footprint: size of the pixel on the surface (metres)
        lod = np.clip(np.log2(np.maximum(footprint * texels_per_metre, 1)), 0, len(levels) - 1)
        level, blend = np.floor(lod).astype(int), (lod - np.floor(lod))[:, None]
        out = np.zeros((len(s), 3))
        for l in np.unique(level):
            m = level == l
            fine, coarse = bilinear(levels[l], s[m], t[m]), bilinear(levels[min(l + 1, len(levels) - 1)], s[m], t[m])
            out[m] = (1 - blend[m]) * fine + blend[m] * coarse
        return out
    return lookup

def render(scene, origins, dirs, fog=400.0, pixel_angle=0.0, pixel_size=0.0, return_distance=False):
    # colour of the nearest surface along every ray (a z-buffer over the objects). A pixel covers
    # pixel_size + distance * pixel_angle metres across the ray (perspective: angle 1/f, orthographic: size 1/scale);
    # on a surface tilted away from the ray it covers more, 1/sqrt(cos) is a compromise between its two sides
    H, W = dirs.shape[:2]
    footprint = lambda dist, cos: (pixel_size + dist * pixel_angle) / np.sqrt(np.maximum(np.abs(cos), 0.02))
    o, d = origins.reshape(-1, 3), dirs.reshape(-1, 3)
    best = np.full(len(d), np.inf)
    up = np.clip(d[:, 1:2], 0, 1)
    sky = np.clip([0.62, 0.78, 0.95] * (0.8 + 0.2 * up) + [0.15, 0.1, 0.0] * (1 - np.clip(4 * up, 0, 1)), 0, 1)
    col = sky.copy()
    for obj in scene:
        if obj[0] == 'rect':                         # textured rectangle: corner p0, edges e1, e2
            _, p0, e1, e2, texture = obj
            n = np.cross(e1, e2)
            n /= np.linalg.norm(n)
            with np.errstate(divide='ignore', invalid='ignore'):
                tt = ((p0 - o) @ n) / (d @ n)
            p = o + tt[:, None] * d
            a, b = (p - p0) @ e1 / (e1 @ e1), (p - p0) @ e2 / (e2 @ e2)
            hit = (tt > 1e-6) & (tt < best) & (a >= 0) & (a <= 1) & (b >= 0) & (b <= 1)
            col[hit] = texture(a[hit] * np.linalg.norm(e1), b[hit] * np.linalg.norm(e2),
                               footprint(tt[hit], d[hit] @ n)) * (0.7 + 0.3 * abs(n @ LIGHT))
            best[hit] = tt[hit]
        elif obj[0] == 'box':                        # axis-aligned box lo..hi: textured walls, flat roof
            _, lo, hi, wall, roof = obj
            oc = o - (lo + hi) / 2                     # only test the rays that pass through the box's bounding sphere
            b = np.einsum('ij,ij->i', oc, d)
            rays = np.flatnonzero(b ** 2 - np.einsum('ij,ij->i', oc, oc) + ((hi - lo) ** 2).sum() / 4 > 0)
            with np.errstate(divide='ignore', invalid='ignore'):
                t1, t2 = (lo - o[rays]) / d[rays], (hi - o[rays]) / d[rays]
            tnear, axis = np.minimum(t1, t2).max(1), np.minimum(t1, t2).argmax(1)
            front = (tnear <= np.maximum(t1, t2).min(1)) & (tnear > 1e-6) & (tnear < best[rays])
            hit, tnear, ax = rays[front], tnear[front], axis[front]
            p = o[hit] + tnear[:, None] * d[hit]
            c = np.tile(np.asarray(roof, float), (len(hit), 1))
            fp = footprint(tnear, d[hit, ax])
            c[ax == 0] = wall(p[ax == 0, 2] - lo[2], p[ax == 0, 1] - lo[1], fp[ax == 0])
            c[ax == 2] = wall(p[ax == 2, 0] - lo[0], p[ax == 2, 1] - lo[1], fp[ax == 2])
            col[hit] = c * (0.55 + 0.45 * np.abs(LIGHT[ax]))[:, None]
            best[hit] = tnear
        elif obj[0] == 'sphere':                     # a beach ball
            _, centre, radius, colours = obj
            oc = o - centre
            b = (oc * d).sum(1)
            disc = b ** 2 - (oc ** 2).sum(1) + radius ** 2
            tt = -b - np.sqrt(np.maximum(disc, 0))
            hit = (disc > 0) & (tt > 1e-6) & (tt < best)
            nrm = (o[hit] + tt[hit, None] * d[hit] - centre) / radius
            segment = np.floor((np.arctan2(nrm[:, 0], nrm[:, 2]) / np.pi + 1) * len(colours) / 2).astype(int) % len(colours)
            c = np.array(colours, float)[segment]
            c[np.abs(nrm[:, 1]) > 0.93] = 0.95
            lambert = np.clip(nrm @ LIGHT, 0, 1)
            col[hit] = np.clip(c * (0.35 + 0.65 * lambert)[:, None] + 0.4 * lambert[:, None] ** 40, 0, 1)
            best[hit] = tt[hit]
    if fog:                                          # far away things fade into the sky
        f = np.exp(-np.minimum(best, 1e6) / fog)[:, None]
        col = f * col + (1 - f) * sky
    if return_distance:                              # distance along each ray to the nearest surface
        return col.reshape(H, W, 3), best.reshape(H, W)
    return col.reshape(H, W, 3)

def photo(scene, K, R, t, width=640, height=480, k1=0.0, ortho=None, fog=400.0, ss=3, depth=False):
    # render with ss x ss rays per pixel (anti-aliasing of the edges), averaged; uint8 RGB
    Ks = K.copy()
    Ks[:2] *= ss
    o, d, valid = camera_rays(Ks, R, t, width * ss, height * ss, k1, ortho * ss if ortho else None)
    pixel = dict(pixel_size=1 / (ortho * ss)) if ortho else dict(pixel_angle=1 / Ks[0, 0])   # footprint of one ray
    img, dist = render(scene, o, d, fog, **pixel, return_distance=True)
    img = cv2.resize((img * valid[..., None] * 255).astype(np.uint8), (width, height), interpolation=cv2.INTER_AREA)
    if depth:                                        # also return the depth Z (camera z) of every pixel's centre ray
        Z = dist * (d @ R[2])
        return img, Z[ss // 2::ss, ss // 2::ss]
    return img

BALL = [(0.9, 0.2, 0.2), (0.95, 0.85, 0.2), (0.2, 0.5, 0.9), (0.95, 0.95, 0.95), (0.2, 0.7, 0.3), (0.95, 0.55, 0.1)]
FACADES = [texture(f'facade{i}', 13, 13) for i in (1, 2, 3)]    # 13 m x 13 m per copy of the photo
ROAD, PAVING, BRICKS = texture('road', 7.5, 7.5), texture('paving', 1.15, 1.15), texture('bricks', 2.4, 1.2)
ROAD_W, GAP, BLDG = 7.5, 1.5, (10.0, 13.0, 12.0)      # road width, pavement, building width x height x depth (m)
X_FACE = ROAD_W / 2 + GAP                             # the buildings' street-facing walls are at x = +-6

def street_scene(n=6):
    # a straight street along z: a 7.5 m road, paved pavements, and n buildings on either side (every 15 m)
    big = 1000.0
    scene = [('rect', np.array([-ROAD_W / 2, 0, -big]), np.array([ROAD_W, 0, 0]), np.array([0, 0, 2 * big]), ROAD),
             ('rect', np.array([-big, 0, -big]), np.array([big - ROAD_W / 2, 0, 0]), np.array([0, 0, 2 * big]), PAVING),
             ('rect', np.array([ROAD_W / 2, 0, -big]), np.array([big, 0, 0]), np.array([0, 0, 2 * big]), PAVING)]
    w, h, d = BLDG
    for i, (x0, x1) in enumerate([(-X_FACE - w, -X_FACE), (X_FACE, X_FACE + w)]):
        for j in range(n):
            z0 = 15.0 * j
            scene.append(('box', np.array([x0, 0, z0]), np.array([x1, h, z0 + d]), FACADES[(i + j) % 3], (0.45, 0.45, 0.47)))
    return scene

def label(img, text, pos=(10, 25), scale=0.6):
    img = img.copy()
    cv2.putText(img, text, pos, cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 4, cv2.LINE_AA)
    cv2.putText(img, text, pos, cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 1, cv2.LINE_AA)
    return img

Helpers for two views. For cameras $[R_1 \mid \mathbf{t}_1]$ and $[R_2 \mid \mathbf{t}_2]$, the pose of camera 2 relative to camera 1 is $R = R_2 R_1^\top$, $\mathbf{t} = \mathbf{t}_2 - R\,\mathbf{t}_1$. Then $E = [\mathbf{t}]_\times R$ and $F = K_2^{-\top} E K_1^{-1}$. The epipoles are the null vectors: $F\mathbf{e} = 0$, $F^\top \mathbf{e}' = 0$.

In [ ]:
def skew(a):
    """[a]x: the matrix with [a]x b = a x b"""
    return np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])

def relative_pose(R1, t1, R2, t2):
    """pose of camera 2 relative to camera 1: X_cam2 = R X_cam1 + t"""
    R = R2 @ R1.T
    return R, t2 - R @ t1

def fundamental(K1, R1, t1, K2, R2, t2):
    """the true F of two known cameras: E = [t]x R in normalised coordinates, F = K2^-T E K1^-1"""
    R, t = relative_pose(R1, t1, R2, t2)
    E = skew(t) @ R
    Fm = np.linalg.inv(K2).T @ E @ np.linalg.inv(K1)
    return Fm / np.linalg.norm(Fm)

def null_point(M):
    """right null vector of M as a homogeneous point (an epipole: F e = 0)"""
    return np.linalg.svd(M)[2][-1]

def line_ends(l, width, height):
    """the two points where the line l = (a, b, c) leaves the image rectangle (or None)"""
    a, b, c = l
    pts = []
    for x in (0, width):                         # left and right border
        if abs(b) > 1e-12:
            y = -(a * x + c) / b
            if 0 <= y <= height:
                pts.append((x, y))
    for y in (0, height):                        # top and bottom border
        if abs(a) > 1e-12:
            x = -(b * y + c) / a
            if 0 <= x <= width:
                pts.append((x, y))
    return pts[:2] if len(pts) >= 2 else None

def draw_line(img, l, colour, thickness=2):
    ends = line_ends(l, img.shape[1], img.shape[0])
    if ends:
        cv2.line(img, tuple(int(round(v)) for v in ends[0]), tuple(int(round(v)) for v in ends[1]), colour, thickness, cv2.LINE_AA)
    return img

def draw_point(img, p, colour, radius=6):
    p = tuple(int(round(v)) for v in p[:2])
    cv2.circle(img, p, radius + 2, (255, 255, 255), -1, cv2.LINE_AA)
    cv2.circle(img, p, radius, colour, -1, cv2.LINE_AA)
    return img

def draw_cross(img, p, colour, size=10, thickness=3):
    x, y = (int(round(v)) for v in p[:2])
    for dx, dy in [(size, size), (size, -size)]:
        cv2.line(img, (x - dx, y - dy), (x + dx, y + dy), (255, 255, 255), thickness + 3, cv2.LINE_AA)
        cv2.line(img, (x - dx, y - dy), (x + dx, y + dy), colour, thickness, cv2.LINE_AA)
    return img

COLOURS = [tuple(int(255 * c) for c in plt.cm.tab10(i)[:3]) for i in range(10)]
hom = lambda p: np.c_[p, np.ones(len(p))]

## A second view

Lecture 14: a sphere moving along a ray, and growing with its distance, gives *exactly* the same image. A second camera 1.5 m to the side sees the sphere move, and its centre moves along a straight line: the *epipolar line* of the first camera's pixel.

In [ ]:
# Lecture 14: a sphere moving along a ray (and growing with its distance) gives camera 1 exactly the same image.
# A second camera 1.5 m to the side sees it move, and its centre moves along a straight line.
wall = ('rect', np.array([-20, -15, 25.0]), np.array([40.0, 0, 0]), np.array([0, 30.0, 0]), BRICKS)
K0 = intrinsics(300, 160, 120)
C1, C2 = np.zeros(3), np.array([1.5, -1.0, 0.0])   # camera 2: 1.5 m to the left, 1 m lower (world: x left, y up)
R1, t1 = look_at(C1, [0, 0, 1])
R2, t2 = look_at(C2, [0.5, -0.5, 9.0])                 # turned a little towards the first camera's view
ray = np.array([0.08, -0.05, 1.0])                     # the sphere's centre moves along this ray
F_sphere = fundamental(K0, R1, t1, K0, R2, t2)

def top_view(s, trail):
    fig, ax = plt.subplots(figsize=(4.8, 2.4), dpi=100)
    ax.plot([25, 25], [-10.5, 10.5], color='0.5', lw=3)
    for C, R, name in [(C1, R1, 'camera 1'), (C2, R2, 'camera 2')]:
        for side in (-1, 1):                           # field of view (half angle 28 degrees)
            d = R.T @ np.array([side * 160 / 300, 0, 1])
            ax.plot([C[2], C[2] + 25 * d[2]], [C[0], C[0] + 25 * d[0]], ':', color='0.6', lw=0.8)
        ax.plot(C[2], C[0], 'ks', ms=5)
    ax.text(-0.5, -1.2, 'camera 1', fontsize=8, ha='left', va='top')
    ax.text(-0.5, 2.4, 'camera 2', fontsize=8, ha='left', va='bottom')
    ax.plot([0, 25], [0, 25 * 0.08], 'C3-', lw=1, alpha=0.6)           # the ray of camera 1
    for s2 in (3, 15):
        ax.add_patch(plt.Circle((s2, 0.08 * s2), 0.12 * s2, color='C3', alpha=0.2))
    ax.add_patch(plt.Circle((s, 0.08 * s), 0.12 * s, color='C3'))
    ax.plot([C2[2], s], [C2[0], 0.08 * s], 'C0-', lw=1.5)              # the ray of camera 2 to the sphere
    ax.text(24.5, -10, 'wall', fontsize=9, ha='right')
    ax.text(1, -10, 'top view', fontsize=9)
    ax.set_xlim(-1.5, 26)
    ax.set_ylim(-10.5, 10.5)
    ax.set_aspect('equal')
    ax.axis('off')
    fig.tight_layout(pad=0.1)
    fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())[..., :3]
    plt.close(fig)
    return cv2.resize(img, (480, 240))

frames, trail = [], []
for s in np.concatenate([np.geomspace(3, 15, 30), np.geomspace(15, 3, 30)]):
    scene = [wall, ('sphere', s * ray, 0.12 * s, BALL)]
    view1 = label(photo(scene, K0, R1, t1, 320, 240), 'camera 1')
    view2 = photo(scene, K0, R2, t2, 320, 240)
    if len(trail) < 30:
        trail.append(project((s * ray)[None], K0, R2, t2)[0])
    for p in trail:                                    # where the centre has been
        cv2.circle(view2, tuple(int(round(v)) for v in p), 3, (0, 0, 0), -1, cv2.LINE_AA)
        cv2.circle(view2, tuple(int(round(v)) for v in p), 2, (255, 255, 255), -1, cv2.LINE_AA)
    view2 = label(view2, 'camera 2')
    gap = np.full((240, 10, 3), 255, np.uint8)
    frames.append(np.concatenate([top_view(s, trail), gap, view1, gap, view2], 1))
save_gif(frames, 'mvg_second_view', duration=70)

## Two cameras looking at the street

Pick a pixel in image 1. Its 3D point lies somewhere on a ray; in image 2 that ray is a line, $\mathbf{l}' = F\mathbf{x}$. With the rendered depth we also know where the true match is, and it lies on that line.

In [ ]:
# The street of Lecture 14, seen by two cameras with known K, R, t (called KS, WS, HS here: the photo cells below use K and H): we know the true F, the epipoles,
# and (from the rendered depth) the true match of every pixel.
WS, HS = 480, 360
KS = intrinsics(320, WS / 2, HS / 2)
street = street_scene(8)
# careful: with look_at (world y up, camera y down) world +x is to the cameras' LEFT, so a camera
# with a larger x is further left

def camera(C, target):
    R, t = look_at(np.asarray(C, float), target)
    return R, t

def backproject(p, Z, KS, R, t):
    """3D world point seen at pixel p with depth Z"""
    Xc = Z * np.linalg.inv(KS) @ np.array([p[0], p[1], 1.0])
    return R.T @ (Xc - t)

def true_matches(pixels, depth1, cam1, cam2, depth2):
    """for pixels of image 1: the 3D points and their pixels in image 2 (None where hidden in image 2)"""
    out = []
    for p in pixels:
        X = backproject(p, depth1[int(p[1]), int(p[0])], KS, *cam1)
        q = project(X[None], KS, *cam2)[0]
        Zc2 = (cam2[0] @ X + cam2[1])[2]
        inside = 0 <= q[0] < WS and 0 <= q[1] < HS
        visible = inside and abs(depth2[int(q[1]), int(q[0])] - Zc2) < 0.02 * Zc2
        out.append((X, q if visible else None))
    return out

def epipolar_pair(cam1, cam2, pixels, markers=()):
    """images 1 and 2 with coloured points in 1, their epipolar lines and true matches in 2, and the epipoles"""
    img1, Z1 = photo(street + list(markers), KS, *cam1, WS, HS, depth=True)
    img2, Z2 = photo(street + list(markers), KS, *cam2, WS, HS, depth=True)
    Fm = fundamental(KS, *cam1, KS, *cam2)
    e1, e2 = null_point(Fm), null_point(Fm.T)       # F e1 = 0, F^T e2 = 0
    matches = true_matches(pixels, Z1, cam1, cam2, Z2)
    hidden = [p for p, (X, q) in zip(pixels, matches) if q is None]
    if hidden:
        print('not visible in image 2:', hidden)
    lines2 = img2.copy()
    for i, p in enumerate(pixels):
        draw_line(lines2, Fm @ [p[0], p[1], 1], COLOURS[i])
    both = lines2.copy()
    left = img1.copy()
    for i, (p, (X, q)) in enumerate(zip(pixels, matches)):
        draw_point(left, p, COLOURS[i])
        if q is not None:
            draw_point(both, q, COLOURS[i])
    # in image 2 all epipolar lines meet at e'. In image 1 (no lines drawn) e is only marked where we could see
    # camera 2, i.e. if it is in front of camera 1
    C2 = -cam2[0].T @ cam2[1]
    front = (cam1[0] @ C2 + cam1[1])[2] > 0
    for img, e, front in [(left, e1, front), (lines2, e2, True), (both, e2, True)]:
        if front and abs(e[2]) > 1e-9 and 0 <= e[0] / e[2] < WS and 0 <= e[1] / e[2] < HS:
            draw_cross(img, e[:2] / e[2], (0, 0, 0))
    return left, lines2, both, Fm, e1, e2

def show_pair(images, titles, name):
    fig, axs = plt.subplots(1, len(images), figsize=(6 * len(images), 4.6))
    for ax, im, title in zip(axs, images, titles):
        ax.imshow(im)
        ax.set_title(title)
        ax.axis('off')
    savefig(name)
    plt.show()

def save_panels(images, name):
    # the same images as separate files, for the slides (they appear one by one)
    if SAVE_FIGURES:
        for i, im in enumerate(images, 1):
            Image.fromarray(im).save(f'figures/{name}_{i}.png')

# pixels in image 1: building corners and windows on both sides of the street, and the road
pixels = [(60, 70), (150, 150), (205, 240), (300, 120), (420, 90), (330, 300)]

def dense_matches(cam1, cam2, n=4000, seed=0, max_depth=200):
    """exact correspondences: random pixels of image 1 (not sky, closer than max_depth) that are visible in image 2"""
    _, Z1 = photo(street, KS, *cam1, WS, HS, depth=True, ss=1)
    _, Z2 = photo(street, KS, *cam2, WS, HS, depth=True, ss=1)
    rng = np.random.default_rng(seed)
    p1 = np.stack([rng.uniform(0, WS, n), rng.uniform(0, HS, n)], 1)
    Z = Z1[p1[:, 1].astype(int), p1[:, 0].astype(int)]
    p1, Z = p1[Z < max_depth], Z[Z < max_depth]
    X = (cam1[0].T @ (Z[:, None] * hom(p1) @ np.linalg.inv(KS).T - cam1[1]).T).T   # back-project with the depth
    p2 = project(X, KS, *cam2)
    Zc2 = (X @ cam2[0].T + cam2[1])[:, 2]
    inside = (p2[:, 0] >= 0) & (p2[:, 0] < WS) & (p2[:, 1] >= 0) & (p2[:, 1] < HS)
    visible = np.zeros(len(p1), bool)
    visible[inside] = np.abs(Z2[p2[inside, 1].astype(int), p2[inside, 0].astype(int)] - Zc2[inside]) < 0.02 * Zc2[inside]
    return p1[visible], p2[visible], X[visible]

### Three problems

Three photos of the street. The same five 3D points (colours) appear in all three. **Correspondence**: which pixels belong together? **Structure**: where are the points in 3D? **Motion**: where are the cameras?

In [ ]:
# Three photos of the street. The same five 3D points (colours) appear in all three: correspondences.
# From them we want the 3D points (structure) and the camera poses (motion).
views = [camera([-2.0, 1.6, -10], [1, 4, 30]), camera([1.0, 2.0, -15], [0, 4, 30]), camera([3.5, 3.0, -11], [-2, 4, 30])]
renders = [photo(street, KS, *cam, WS, HS, depth=True) for cam in views]
seeds = [(70, 90), (140, 230), (330, 110), (400, 200), (250, 250)]     # pixels in view 1
points3d = [backproject(p, renders[0][1][p[1], p[0]], KS, *views[0]) for p in seeds]
images = []
for (img, Z), cam in zip(renders, views):
    img = img.copy()
    for i, X in enumerate(points3d):
        q = project(X[None], KS, *cam)[0]
        Zc = (cam[0] @ X + cam[1])[2]
        if 0 <= q[0] < WS and 0 <= q[1] < HS and abs(Z[int(q[1]), int(q[0])] - Zc) < 0.02 * Zc:
            draw_point(img, q, COLOURS[i])
        else:
            print('hidden:', i)
    images.append(img)

def street_top_view(ax, cams, points, zmax=40):
    w, h, d = BLDG
    for x0 in (-X_FACE - w, X_FACE):
        for j in range(3):
            ax.add_patch(plt.Rectangle((x0, 15 * j), w, d, color='0.8'))
    for i, (R, t) in enumerate(cams):
        C = -R.T @ t
        for side in (-1, 1):
            dvec = R.T @ np.array([side * (WS / 2) / KS[0, 0], 0, 1])
            ax.plot([C[0], C[0] + 12 * dvec[0]], [C[2], C[2] + 12 * dvec[2]], color='0.4', lw=0.8)
        ax.plot(C[0], C[2], 'ks', ms=6)
        ax.text(C[0] - 0.8, C[2] - 1.5, f'{i + 1}', fontsize=11)
    for i, X in enumerate(points):
        ax.plot(X[0], X[2], 'o', color=np.array(COLOURS[i]) / 255, ms=8, mec='white')
    ax.set_xlim(18, -18)                               # world x points to the cameras' left: flip it
    ax.set_ylim(-18, zmax)
    ax.set_aspect('equal')
    ax.axis('off')

fig, axs = plt.subplots(1, 4, figsize=(22, 4.6), gridspec_kw=dict(width_ratios=[1, 1, 1, 0.55]))
for i, (ax, im) in enumerate(zip(axs, images)):
    ax.imshow(im)
    ax.set_title(f'camera {i + 1}')
    ax.axis('off')
street_top_view(axs[3], views, points3d)
axs[3].set_title('top view')
savefig('mvg_three_views')
plt.show()
save_panels(images, 'mvg_three_views')
if SAVE_FIGURES:
    fig, ax = plt.subplots(figsize=(3.2, 3.6))
    street_top_view(ax, views, points3d)
    fig.savefig('figures/mvg_three_views_top.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

### Epipolar lines for three kinds of camera motion

* **General motion**: all epipolar lines meet at the epipole $\mathbf{e}'$, the image of camera 1's centre (the grey ball).
* **Sideways** (parallel to the image plane): the epipoles are at infinity, the epipolar lines are parallel (and horizontal: image rows).
* **Forward** (along the optical axis): the epipole is the principal point, the *focus of expansion*; points move away from it.

In [ ]:
# Three kinds of camera motion. A grey ball marks camera 1's centre: camera 2 sees it exactly at the epipole e'.
CAMERA_BALL = [(0.35, 0.35, 0.38)] * 2
cases = {
    'general':  (camera([-1.5, 1.6, -8], [2, 4, 30]), camera([3.0, 2.5, -18], [-1, 4, 30])),
    'sideways': (camera([-1.0, 1.6, -10], [-1.0, 4, 30]), camera([1.0, 1.6, -10], [1.0, 4, 30])),
    'forward':  (camera([0.0, 1.6, -14], [0, 1.6, 30]), camera([0.0, 1.6, -8], [0, 1.6, 30])),
}
case_pixels = {
    'general':  [(60, 70), (150, 150), (205, 250), (300, 120), (420, 90), (330, 300)],
    'sideways': [(60, 70), (110, 200), (205, 250), (300, 120), (360, 60), (330, 300)],
    'forward':  [(150, 100), (190, 160), (215, 212), (290, 150), (320, 90), (262, 218)],
}
pairs = {}
for name, (cam1, cam2) in cases.items():
    C1 = -cam1[0].T @ cam1[1]
    ball = [('sphere', C1, 0.6, CAMERA_BALL)] if name == 'general' else []
    left, lines2, both, Fm, e1, e2 = epipolar_pair(cam1, cam2, case_pixels[name], ball)
    pairs[name] = (left, lines2, both, Fm)
    print(f'{name:9s} e  = {np.round(e1 / e1[2], 1) if abs(e1[2]) > 1e-9 else "at infinity, direction " + str(np.round(e1[:2] / np.linalg.norm(e1[:2]), 2))}')
    print(f'{"":9s} e\' = {np.round(e2 / e2[2], 1) if abs(e2[2]) > 1e-9 else "at infinity, direction " + str(np.round(e2[:2] / np.linalg.norm(e2[:2]), 2))}')
    show_pair([left, both], ['image 1', "image 2: epipolar lines and true matches"], f'mvg_motion_{name}')
    save_panels([left, lines2, both], f'mvg_motion_{name}')

# the two cameras of each case from above (for the slides: next to the images)
if SAVE_FIGURES:
    for name, cams in cases.items():
        fig, ax = plt.subplots(figsize=(2.6, 4.2))
        street_top_view(ax, cams, [], zmax=32)
        ax.set_ylim(-22, 32)
        fig.savefig(f'figures/mvg_motion_{name}_top.png', dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig)

In [ ]:
scale = 0.2   # the originals are 6016x4016
image1 = cv2.cvtColor(cv2.resize(cv2.imread('image_cc_1.jpg'), (0, 0), fx=scale, fy=scale, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
image2 = cv2.cvtColor(cv2.resize(cv2.imread('image_cc_2.jpg'), (0, 0), fx=scale, fy=scale, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)

# SIFT matches with the ratio test (Lecture 5)
sift = cv2.SIFT_create(4000)
kp1, des1 = sift.detectAndCompute(cv2.cvtColor(image1, cv2.COLOR_RGB2GRAY), None)
kp2, des2 = sift.detectAndCompute(cv2.cvtColor(image2, cv2.COLOR_RGB2GRAY), None)
matches = [m for m, n in cv2.BFMatcher().knnMatch(des1, des2, k=2) if m.distance < 0.8 * n.distance]
pts1 = np.float64([kp1[m.queryIdx].pt for m in matches])
pts2 = np.float64([kp2[m.trainIdx].pt for m in matches])
print(f'{len(matches)} matches')

fig, axs = plt.subplots(1, 2, figsize=(20, 7))
for ax, im in zip(axs, [image1, image2]):
    ax.imshow(im)
    ax.axis('off')
axs[0].set_title('Image 1')
axs[1].set_title('Image 2')
plt.show()

## Plane and parallax

Most matches lie on the façade, which is (almost) a plane: a single homography maps them from one image to the other. Points *off* the plane (trees, lamp posts, the ground) do not follow the homography. Their residual displacement, the **parallax**, is what tells us about the 3D structure and the epipolar geometry.

We estimate both models robustly. For the fundamental matrix we use MAGSAC, a RANSAC variant (Practical 1) that copes better with this situation.

In [ ]:
H, h_inliers = cv2.findHomography(pts1, pts2, cv2.RANSAC, 2.0)
F_cv, f_inliers = cv2.findFundamentalMat(pts1, pts2, cv2.USAC_MAGSAC, 1.0, 0.999, 10000)
h_inliers, f_inliers = h_inliers.ravel() > 0, f_inliers.ravel() > 0
plane = f_inliers & h_inliers
parallax = f_inliers & ~h_inliers
print(f'homography inliers: {h_inliers.sum()}, fundamental matrix inliers: {f_inliers.sum()}, off the plane: {parallax.sum()}')

# where the homography sends the points of image 1, and how far off the plane points are
transferred = cv2.perspectiveTransform(pts1[:, None], H)[:, 0]
fig, axs = plt.subplots(1, 2, figsize=(20, 7))
axs[0].imshow(image1)
axs[0].scatter(*pts1[plane].T, s=6, c='deepskyblue', label='on the plane (homography inlier)')
axs[0].scatter(*pts1[parallax].T, s=20, c='red', label='off the plane')
axs[0].legend(loc='lower left')
axs[0].set_title('Image 1: matches consistent with the epipolar geometry')
axs[1].imshow(image2)
axs[1].quiver(*transferred[parallax].T, *(pts2[parallax] - transferred[parallax]).T, color='red', angles='xy', scale_units='xy', scale=1, width=0.002)
axs[1].set_xlim(0, image2.shape[1])
axs[1].set_ylim(image2.shape[0], 0)
axs[1].set_title('Image 2: parallax = true position - position predicted by the plane')
for ax in axs:
    ax.axis('off')
savefig('mvg_plane_parallax')
plt.show()

## Epipolar lines

A point $\mathbf{x}_1$ in image 1 must lie on the line $\mathbf{l}_2 = F \mathbf{x}_1$ in image 2 (and $\mathbf{x}_2$ on $\mathbf{l}_1 = F^\top \mathbf{x}_2$ in image 1). All epipolar lines pass through the epipoles, the null vectors of $F$ and $F^\top$.

In [ ]:
def draw_lines(ax, lines, image, colors, linewidth=1):
    w = image.shape[1]
    for (a, b, c), color in zip(lines, colors):
        ax.plot([0, w], [-c / b, -(c + a * w) / b], '-', color=color, linewidth=linewidth)
    ax.set_xlim(0, w)
    ax.set_ylim(image.shape[0], 0)

def epipole(F):
    e = np.linalg.svd(F)[2][-1]
    return e[:2] / e[2]

hom = lambda p: np.c_[p, np.ones(len(p))]
rng = np.random.default_rng(1)
idx = rng.choice(np.where(parallax)[0], 6, replace=False).tolist() + rng.choice(np.where(plane)[0], 6, replace=False).tolist()
colors = plt.cm.tab20(np.linspace(0, 1, len(idx)))
fig, axs = plt.subplots(1, 2, figsize=(20, 7))
axs[0].imshow(image1)
draw_lines(axs[0], hom(pts2[idx]) @ F_cv, image1, colors)
axs[0].scatter(*pts1[idx].T, c=colors, s=60, edgecolors='white', zorder=3)
axs[1].imshow(image2)
draw_lines(axs[1], hom(pts1[idx]) @ F_cv.T, image2, colors)
axs[1].scatter(*pts2[idx].T, c=colors, s=60, edgecolors='white', zorder=3)
for ax, F_ in zip(axs, [F_cv, F_cv.T]):
    ax.set_title(f'epipole at ({epipole(F_)[0]:.0f}, {epipole(F_)[1]:.0f}), outside the image')
    ax.axis('off')
savefig('mvg_epipolar_lines')
plt.show()

## The eight-point algorithm

Every correspondence gives one linear equation $\mathbf{x}_2^\top F \mathbf{x}_1 = 0$ in the 9 entries of $F$. Stack them into $A \mathbf{f} = 0$ and take the right singular vector of $A$ with the smallest singular value. Two details matter in practice:

* **Normalisation** (Hartley): pixel coordinates are in the hundreds, so the entries of $A$ range over many orders of magnitude and the least-squares problem is badly conditioned. Translate and scale the points so that they are centred with an average distance of $\sqrt{2}$, solve, then undo the transformation.
* **Rank 2**: the solution of the linear system is generally not singular, so its epipolar lines do not meet in one point. Set the smallest singular value to zero.

We measure the quality by the *symmetric epipolar distance*: how far (in pixels) each point is from the epipolar line of its match.

In [ ]:
def normalisation(p):
    c = p.mean(0)
    s = np.sqrt(2) / np.linalg.norm(p - c, axis=1).mean()
    return np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1]])

def eight_point(p1, p2, normalise=True, rank2=True):
    T1, T2 = (normalisation(p1), normalisation(p2)) if normalise else (np.eye(3), np.eye(3))
    x1, x2 = hom(p1) @ T1.T, hom(p2) @ T2.T
    A = np.stack([x2[:, 0] * x1[:, 0], x2[:, 0] * x1[:, 1], x2[:, 0],
                  x2[:, 1] * x1[:, 0], x2[:, 1] * x1[:, 1], x2[:, 1],
                  x1[:, 0], x1[:, 1], np.ones(len(x1))], axis=1)
    F = np.linalg.svd(A)[2][-1].reshape(3, 3)
    if rank2:
        U, S, Vt = np.linalg.svd(F)
        F = U @ np.diag([S[0], S[1], 0]) @ Vt
    F = T2.T @ F @ T1
    return F / np.linalg.norm(F)

def epipolar_distance(F, p1, p2):
    l2, l1 = hom(p1) @ F.T, hom(p2) @ F
    d2 = np.abs((hom(p2) * l2).sum(1)) / np.hypot(l2[:, 0], l2[:, 1])
    d1 = np.abs((hom(p1) * l1).sum(1)) / np.hypot(l1[:, 0], l1[:, 1])
    return (d1 + d2) / 2

# 1) fit on all inliers: with many accurate points every variant works
p1, p2 = pts1[f_inliers], pts2[f_inliers]
for normalise in [False, True]:
    for rank2 in [False, True]:
        d = epipolar_distance(eight_point(p1, p2, normalise, rank2), p1, p2)
        print(f'all {len(p1)} inliers   normalised: {normalise!s:5}  rank 2: {rank2!s:5}  epipolar distance: median {np.median(d):.2f} px')

# with few, noisy correspondences normalisation matters: measured against the true F on the street below

### How many RANSAC samples?

Real correspondences contain outliers, so $F$ is estimated inside RANSAC from minimal samples. To draw at least one sample of $s$ inliers with probability $p$ when a fraction $w$ of the matches are inliers, RANSAC needs $N = \log(1 - p) / \log(1 - w^s)$ samples. Fewer points per sample help a lot: this is one reason for the seven-point and (with known $K$) the five-point algorithm.

In [ ]:
# How many RANSAC samples (Lecture 5, Practical 1)? To draw at least one all-inlier sample with probability p,
# N = log(1 - p) / log(1 - w^s) samples are needed (inlier ratio w, sample size s). Smaller minimal samples help a lot.
p = 0.99
w = np.linspace(0.25, 0.95, 200)
fig, ax = plt.subplots(figsize=(6.4, 4.2))
for s, name, c in [(8, 'eight-point', 'C3'), (7, 'seven-point', 'C1'), (5, 'five-point (calibrated, E)', 'C2')]:
    ax.semilogy(100 * w, np.log(1 - p) / np.log(1 - w ** s), lw=2.5, color=c, label=f'{s} points: {name}')
    n50 = np.log(1 - p) / np.log(1 - 0.5 ** s)
    ax.plot(50, n50, 'o', color=c)
    ax.annotate(f'{n50:.0f}', (50, n50), (6, 4), textcoords='offset points', color=c, fontsize=12)
    print(f'{s} points, 50% inliers, 99% confidence: {n50:.0f} samples')
ax.set_xlabel('inliers (%)', fontsize=12)
ax.set_ylabel('samples needed (99% confidence)', fontsize=12)
ax.grid(alpha=0.3, which='both')
ax.legend(fontsize=11)
savefig('mvg_ransac_samples')
plt.show()

### Why normalise? Measured against the true F

On the Christ Church photos we do not know the true $F$. On the rendered street we do. We repeat the eight-point algorithm 200 times on 20 exact correspondences plus 1 px of noise and measure how far all true matches are from the estimated epipolar lines.

Without normalisation, the entries of $A$ range from 1 to about $10^5$ ($x'x$ with pixel coordinates in the hundreds), and the singular values of $A$ spread over five orders of magnitude: noise in the small entries is amplified.

In [ ]:
# Why normalise? On the rendered street pair we know the true F, so we can measure the error exactly.
# 200 times: 20 exact correspondences + 1 px of noise -> eight-point with and without normalisation;
# error = median distance of all true matches to the estimated epipolar lines.
cam1, cam2 = cases['general']
m1, m2, _ = dense_matches(cam1, cam2)
F_street = fundamental(KS, *cam1, KS, *cam2)
rng = np.random.default_rng(0)
n, sigma = 20, 1.0
trials = []
for _ in range(200):
    sub = rng.choice(len(m1), n, replace=False)
    a, b = m1[sub] + rng.normal(0, sigma, (n, 2)), m2[sub] + rng.normal(0, sigma, (n, 2))
    trials.append((a, b, [np.median(epipolar_distance(eight_point(a, b, norm), m1, m2)) for norm in (False, True)]))
err = np.array([e for _, _, e in trials])
print(f'{len(m1)} true matches; median error over 200 trials: without normalisation {np.median(err[:, 0]):.2f} px, '
      f'with {np.median(err[:, 1]):.2f} px; 90th percentile {np.percentile(err[:, 0], 90):.1f} vs {np.percentile(err[:, 1], 90):.1f} px')

# the condition number of A: how much noise in the data can be amplified in the solution
a, b, _ = trials[0]
for norm in (False, True):
    T1, T2 = (normalisation(a), normalisation(b)) if norm else (np.eye(3), np.eye(3))
    x1, x2 = hom(a) @ T1.T, hom(b) @ T2.T
    A = np.stack([x2[:, 0] * x1[:, 0], x2[:, 0] * x1[:, 1], x2[:, 0], x2[:, 1] * x1[:, 0], x2[:, 1] * x1[:, 1], x2[:, 1],
                  x1[:, 0], x1[:, 1], np.ones(n)], 1)
    s = np.linalg.svd(A)[1]
    print(f'normalised: {norm!s:5}  entries of A from {np.abs(A).min():.1e} to {np.abs(A).max():.1e}, '
          f'singular values s1 / s8 = {s[0] / s[7]:.1e}')

# one trial where it goes wrong (the 80th percentile of the unnormalised errors), drawn in image 2
k = int(np.argsort(err[:, 0])[int(0.8 * len(err))])
a, b, e = trials[k]
img2 = photo(street, KS, *cam2, WS, HS)
shown = m1[np.linspace(0, len(m1) - 1, 8).astype(int)]
panels = []
for norm, colour in [(False, (220, 40, 40)), (True, (30, 160, 60))]:
    im = img2.copy()
    Fe = eight_point(a, b, norm)
    for p in shown:
        draw_line(im, F_street @ [p[0], p[1], 1], (255, 255, 255), 3)      # true epipolar line (white)
        draw_line(im, Fe @ [p[0], p[1], 1], colour, 2)                   # estimated
    panels.append(label(im, f'{"normalised" if norm else "not normalised"}: {e[norm]:.1f} px', (10, 25), 0.7))

fig, axs = plt.subplots(1, 3, figsize=(20, 5), gridspec_kw=dict(width_ratios=[1, 1, 0.9]))
for ax, im in zip(axs, panels):
    ax.imshow(im)
    ax.axis('off')
axs[0].set_title('without normalisation (white: true lines)')
axs[1].set_title('with normalisation')
bins = np.geomspace(0.3, 100, 30)
axs[2].hist(np.clip(err[:, 0], 0.3, 100), bins, alpha=0.6, color='C3', label='not normalised')
axs[2].hist(np.clip(err[:, 1], 0.3, 100), bins, alpha=0.6, color='C2', label='normalised')
axs[2].set_xscale('log')
axs[2].set_xlabel('error: median distance to the epipolar lines (px)')
axs[2].set_ylabel('trials')
axs[2].legend()
axs[2].set_title(f'200 trials, {n} matches, {sigma:.0f} px noise')
savefig('mvg_normalisation')
plt.show()
if SAVE_FIGURES:
    save_panels(panels, 'mvg_normalisation')
    fig, ax = plt.subplots(figsize=(5.2, 3.9))
    ax.hist(np.clip(err[:, 0], 0.3, 100), bins, alpha=0.6, color='C3', label='not normalised')
    ax.hist(np.clip(err[:, 1], 0.3, 100), bins, alpha=0.6, color='C2', label='normalised')
    ax.set_xscale('log')
    ax.set_xlabel('median distance to the true lines (px)')
    ax.set_ylabel('trials (of 200)')
    ax.legend()
    fig.savefig('figures/mvg_normalisation_hist.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

# What normalisation does to the points (one trial's 20 points in image 1)
a, b, _ = trials[0]
an = (hom(a) @ normalisation(a).T)[:, :2]
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
axs[0].add_patch(plt.Rectangle((0, 0), WS, HS, fill=False, ec='0.5'))
axs[0].scatter(*a.T, s=30, c='C0')
axs[0].plot(0, 0, 'k+', ms=14, mew=2)
axs[0].set_xlim(-30, WS + 30)
axs[0].set_ylim(HS + 30, -30)
axs[0].set_title(f'pixels: x up to {WS}, so x′x up to {WS * WS / 1e5:.1f}·10⁵', fontsize=13)
axs[1].add_patch(plt.Circle((0, 0), np.sqrt(2), fill=False, ls='--', ec='C1', lw=1.5))
axs[1].scatter(*an.T, s=30, c='C0')
axs[1].plot(0, 0, 'k+', ms=14, mew=2)
axs[1].set_xlim(-2.6, 2.6)
axs[1].set_ylim(2.2, -2.2)
axs[1].set_title('normalised: mean 0, mean distance √2 (dashed)', fontsize=13)
for ax in axs:
    ax.set_aspect('equal')
savefig('mvg_normalisation_points')
plt.show()

# Where the estimate is wrong: every true match in image 2, coloured by its distance to its estimated epipolar line
# (the bad trial from above). Black crosses: the 20 matches the estimate was computed from.
fig, axs = plt.subplots(1, 2, figsize=(13, 4.6))
a, b, e = trials[k]
for ax, norm in zip(axs, (False, True)):
    Fe = eight_point(a, b, norm)
    l2 = hom(m1) @ Fe.T
    d = np.abs((hom(m2) * l2).sum(1)) / np.hypot(l2[:, 0], l2[:, 1])
    ax.imshow((0.45 * img2 + 0.55 * 255).astype(np.uint8))
    sc = ax.scatter(*m2.T, c=np.minimum(d, 10), cmap='inferno_r', vmin=0, vmax=10, s=4)
    ax.scatter(*b.T, marker='x', c='k', s=40, lw=2)
    ax.set_title(f'{"normalised" if norm else "not normalised"}: median {np.median(d):.1f} px', fontsize=14)
    ax.axis('off')
cb = fig.colorbar(sc, ax=axs, fraction=0.025, pad=0.02)
cb.set_label('distance to the epipolar line (px)', fontsize=12)
savefig('mvg_normalisation_errmap')
plt.show()

In [ ]:
# the rank-2 constraint (Lecture slide "Enforcing rank-2 constraint"): epipolar lines of the
# initial (full-rank) estimate do not meet in a single point; after rank-2 projection they do.
# The effect is easiest to see with only a few, noisy correspondences.
rng = np.random.default_rng(3)
sub = rng.choice(len(p1), 12, replace=False)
noisy1, noisy2 = p1[sub] + rng.normal(0, 2, (12, 2)), p2[sub] + rng.normal(0, 2, (12, 2))
F_rank2 = eight_point(noisy1, noisy2, normalise=True, rank2=True)
e = epipole(F_rank2.T)                                 # the epipole in image 2
shown = hom(p1[rng.choice(len(p1), 15, replace=False)])
fig, axs = plt.subplots(1, 2, figsize=(20, 8))
for ax, rank2 in zip(axs, [False, True]):
    F_ = eight_point(noisy1, noisy2, normalise=True, rank2=rank2)
    for a, b, c in shown @ F_.T:
        xs = np.array([e[0] - 1000, e[0] + 1000])
        ax.plot(xs, -(c + a * xs) / b, '-', linewidth=1)
    ax.plot(*e, 'k+', markersize=20)
    ax.set_xlim(e[0] - 60, e[0] + 60)                  # zoom in around the epipole
    ax.set_ylim(e[1] + 60, e[1] - 60)
    ax.set_aspect('equal')
    ax.set_title(('rank-2 estimate: all lines meet at the epipole' if rank2 else 'initial estimate (rank 3): lines do not meet in one point') + '\n(120x120 px around the epipole of image 2)')
savefig('mvg_rank2')
plt.show()

## Degenerate configuration: a plane

If all points lie on a plane, the correspondences are explained by a homography $H$ and the eight-point system no longer has a unique solution: $F = [\mathbf{e}_2]_\times H$ fits for *any* epipole $\mathbf{e}_2$. We can see this in the singular values of $A$: with points on the plane only, three of them are (close to) zero.

In [ ]:
def design_matrix(p1, p2):
    T1, T2 = normalisation(p1), normalisation(p2)
    x1, x2 = hom(p1) @ T1.T, hom(p2) @ T2.T
    return np.stack([x2[:, 0] * x1[:, 0], x2[:, 0] * x1[:, 1], x2[:, 0], x2[:, 1] * x1[:, 0], x2[:, 1] * x1[:, 1], x2[:, 1], x1[:, 0], x1[:, 1], np.ones(len(x1))], axis=1)

plt.figure(figsize=(8, 4))
for name, mask in [('points on the façade only', plane), ('all inliers (with parallax)', f_inliers)]:
    s = np.linalg.svd(design_matrix(pts1[mask], pts2[mask]))[1]
    plt.semilogy(range(1, 10), s / s[0], 'o-', label=name)
plt.xlabel('index of the singular value of A')
plt.ylabel('relative singular value')
plt.legend()
plt.grid(alpha=0.3)
savefig('mvg_degenerate_plane')
plt.show()

### The same on the rendered street: the ground is an exact plane

The façade above is only roughly a plane (bay windows, the tower), so its "off-plane" points are hard to interpret. On the street the ground (road and pavements, $y = 0$) is an exact plane. Its homography $H$ moves the ground points correctly; for points on the buildings $H$ guesses wrong, and the error (the parallax) points along the epipolar lines, through $\mathbf{e}'$.

With ground points only, $F = [\mathbf{e}']_\times H$ fits every correspondence for *any* $\mathbf{e}'$: three different choices, all consistent with the ground, only one with the buildings.

In [ ]:
# Plane and parallax on the rendered street, where the plane is exact: the ground (road and pavements, y = 0).
# two photos 3 m apart along the street (and 1 m to the side), same viewing direction: short parallax arrows
cam1, cam2 = camera([0.0, 1.6, -10], [0.0, 3, 30]), camera([-1.0, 1.6, -7], [-1.0, 3, 33])
g1, g2 = photo(street, KS, *cam1, WS, HS), photo(street, KS, *cam2, WS, HS)
q1, q2, Xq = dense_matches(cam1, cam2, n=8000, seed=5)
ground = np.abs(Xq[:, 1]) < 0.01                                   # back-projected from the depth: y = 0 up to rounding
H_ground, _ = cv2.findHomography(q1[ground], q2[ground])          # exact matches: no RANSAC needed
F_true = fundamental(KS, *cam1, KS, *cam2)
e_true = null_point(F_true.T)
e_true = e_true[:2] / e_true[2]
print(f'{ground.sum()} matches on the ground, {(~ground).sum()} on the buildings; e\' = {e_true.round(1)}')

rng = np.random.default_rng(4)
on = rng.choice(np.flatnonzero(ground), 30, replace=False)
# facade points low on the buildings, whose parallax (the ground plane's prediction -> true match) stays in view
pred_all = cv2.perspectiveTransform(q1[:, None], H_ground)[:, 0]                 # where the ground plane puts each point
shift = np.linalg.norm(pred_all - q2, axis=1)
inside = (pred_all[:, 0] > 5) & (pred_all[:, 0] < WS - 5) & (pred_all[:, 1] > 5) & (pred_all[:, 1] < HS - 5)
off = rng.choice(np.flatnonzero(~ground & (Xq[:, 1] < 2.5) & (shift > 15) & (shift < 120) & inside), 12, replace=False)
predicted = pred_all[off]
BLUE, RED = (40, 120, 230), (220, 40, 40)

def dimmed(img):
    return (0.6 * img + 0.4 * 255).astype(np.uint8)

im1 = dimmed(g1)
for p in q1[on]:
    draw_point(im1, p, BLUE, 5)
for p in q1[off]:
    draw_point(im1, p, RED, 5)
im2 = dimmed(g2)
for p in q2[off]:                                       # the epipolar line of each facade point: through e'
    draw_line(im2, np.cross([*e_true, 1], [*p, 1]), (90, 90, 90), 1)
for p in q2[on]:
    draw_point(im2, p, BLUE, 5)
for p, s in zip(q2[off], predicted):
    cv2.arrowedLine(im2, tuple(int(v) for v in s), tuple(int(v) for v in p), RED, 2, cv2.LINE_AA, tipLength=0.15)
    draw_point(im2, s, (255, 255, 255), 4)
draw_cross(im2, e_true, (0, 0, 0))
show_pair([im1, im2], ['image 1: ground (blue), buildings (red)', "image 2: H moves the ground points correctly; arrows: H's guess -> true match"], 'mvg_plane_parallax_street')
save_panels([im1, im2], 'mvg_plane_parallax_street')

# Degenerate: from ground points alone, F = [e']x H fits every match for ANY e'. Three choices of e', each fits
# the ground points exactly; only the true one also fits the buildings.
U_ground = design_matrix(q1[ground], q2[ground])
s = np.linalg.svd(U_ground)[1]
print('singular values of U, ground points only:', (s / s[0]).round(6))
show_on, show_off = on[:5], off[:5]
panels = []
for e_ in [e_true, np.array([60.0, 300.0]), np.array([420.0, 70.0])]:
    F_ = skew(np.array([*e_, 1])) @ H_ground
    im = dimmed(g2)
    for idx, colour in [(show_on, BLUE), (show_off, RED)]:
        for i in idx:
            draw_line(im, F_ @ [*q1[i], 1], colour, 2)
            draw_point(im, q2[i], colour, 6)
    draw_cross(im, e_, (0, 0, 0))
    res_on = np.median(epipolar_distance(F_, q1[ground], q2[ground]))
    res_off = np.median(epipolar_distance(F_, q1[~ground], q2[~ground]))
    print(f"e' = {e_.round(1)}: median distance ground {res_on:.3f} px, buildings {res_off:.1f} px")
    panels.append(im)
show_pair(panels, ['true e\'', 'another e\'', 'another e\''], 'mvg_degenerate_street')
save_panels(panels, 'mvg_degenerate_street')

## Two-view reconstruction

With the intrinsics $K$ we get the essential matrix $E = K^\top F K$, which decomposes into the rotation and the (unit-length) translation between the cameras. The focal length is in the photos' EXIF data: 24 mm on a full-frame (36 mm wide) sensor. Then we **triangulate** every correspondence: find the 3D point whose projections are closest to the two observations. The reconstruction is only defined up to scale.

In [ ]:
h, w = image1.shape[:2]
f = 24 / 36 * w                 # 24 mm lens, 36 mm wide sensor: f in pixels
print(f'image width {w} px -> f = 24 / 36 * {w} = {f:.0f} px')
K = np.array([[f, 0, w / 2], [0, f, h / 2], [0, 0, 1]])
E = K.T @ F_cv @ K
# E decomposes into four (R, t) candidates; only one puts the points in front of both cameras
def triangulate(R, t):
    P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
    P2 = K @ np.hstack([R, t])
    Xh = cv2.triangulatePoints(P1, P2, pts1[f_inliers].T, pts2[f_inliers].T)
    X = (Xh[:3] / Xh[3]).T
    return X, (X[:, 2] > 0) & ((R @ X.T + t)[2] > 0), P1

R1, R2, t0 = cv2.decomposeEssentialMat(E)
candidates = [(R1, t0), (R1, -t0), (R2, t0), (R2, -t0)]
counts = [triangulate(R_, t_)[1].sum() for R_, t_ in candidates]
for (R_, t_), c in zip(candidates, counts):
    print(f'rotation {np.degrees(np.arccos(np.clip((np.trace(R_) - 1) / 2, -1, 1))):5.1f} deg, t = {t_.ravel().round(2)}: {c} points in front of both cameras')
R, t = candidates[int(np.argmax(counts))]
X, in_front, P1 = triangulate(R, t)
reproj = (P1 @ np.c_[X, np.ones(len(X))].T)
reproj = (reproj[:2] / reproj[2]).T
print(f'reprojection error (image 1): median {np.median(np.linalg.norm(reproj - pts1[f_inliers], axis=1)):.2f} px')

colors = image1[pts1[f_inliers][:, 1].astype(int), pts1[f_inliers][:, 0].astype(int)] / 255
on_plane = plane[f_inliers]
p_in = pts1[f_inliers][in_front]
X, colors, on_plane = X[in_front], colors[in_front], on_plane[in_front]
lim = np.percentile(X[:, 2], 98)
keep = X[:, 2] < lim

# image 1 with every reconstructed point coloured by its depth (in units of the unknown baseline)
fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow((0.55 * image1 + 0.45 * 255).astype(np.uint8))
sc = ax.scatter(*p_in[keep].T, c=X[keep, 2], cmap='turbo', s=10)
cb = fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
cb.set_label('depth (baseline = 1)', fontsize=13)
ax.axis('off')
ax.set_title('Image 1: triangulated depth of each match', fontsize=14)
savefig('mvg_twoview_depth')
plt.show()

# the same points from above, with the two cameras
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X[keep, 0], X[keep, 2], c=colors[keep], s=10)
ax.scatter(X[keep & ~on_plane, 0], X[keep & ~on_plane, 2], facecolors='none', edgecolors='red', s=50, label='off the façade plane')
c2 = -R.T @ t.ravel()
for C_, R_, name, col in [(np.zeros(3), np.eye(3), 'camera 1', 'k'), (c2, R, 'camera 2', 'C0')]:
    d_ = R_.T @ np.array([0, 0, 1.0])
    ax.plot(C_[0], C_[2], '^', color=col, ms=14, label=name)
    ax.annotate('', (C_[0] + 3 * d_[0], C_[2] + 3 * d_[2]), (C_[0], C_[2]), arrowprops=dict(arrowstyle='->', color=col, lw=2))
ax.set_xlabel('x (baseline = 1)', fontsize=13)
ax.set_ylabel('z: depth (baseline = 1)', fontsize=13)
ax.set_aspect('equal')
ax.legend(fontsize=12, loc='lower right')
ax.set_title('From above: the façade is a line, trees and lamps in front', fontsize=14)
savefig('mvg_twoview_top')
plt.show()

### Four solutions for R and t

$E = [\mathbf{t}]_\times R$ fixes $R$ and the *direction* of $\mathbf{t}$, but four $(R, \mathbf{t})$ are consistent with it: $\mathbf{t}$ or $-\mathbf{t}$, and $R$ or $R$ rotated by 180° about the baseline. Triangulating with each one, only one puts the points in front of both cameras. On the street pair we can check the result against the true pose.

In [ ]:
# From E back to the cameras. E = [t]x R gives R and the direction of t, but four (R, t) are consistent with it:
# t or -t, and R or R rotated by 180 degrees about the baseline. Only one puts the points in front of both cameras.
cam1, cam2 = cases['general']
rng = np.random.default_rng(1)
sub = rng.choice(len(m1), 200, replace=False)
a, b = m1[sub] + rng.normal(0, 0.5, (200, 2)), m2[sub] + rng.normal(0, 0.5, (200, 2))
E_est = KS.T @ eight_point(a, b) @ KS                       # both cameras have the same K
R_a, R_b, t_unit = cv2.decomposeEssentialMat(E_est)
R_true, t_true = relative_pose(*cam1, *cam2)
def draw_camera(ax, C, R_, name, colour='k'):
    # a camera from above: its centre and a triangle for the field of view (+z is its viewing direction)
    corners = [C + R_.T @ np.array([sx * 0.18, 0, 0.3]) for sx in (-1, 1)]
    ax.fill([C[0], corners[0][0], corners[1][0]], [C[2], corners[0][2], corners[1][2]], color=colour, alpha=0.25)
    ax.plot([corners[0][0], C[0], corners[1][0]], [corners[0][2], C[2], corners[1][2]], color=colour, lw=1.5)
    ax.plot(C[0], C[2], 'o', color=colour, ms=7)
    ax.text(C[0] + 0.1, C[2] - 0.3, name, fontsize=19, color=colour)

fig, axs = plt.subplots(2, 2, figsize=(9, 10.5), sharex=True, sharey=True)
in_front = []
names = ['R₁, +t', 'R₁, −t', 'R₂, +t', 'R₂, −t']
for ax, (R_, t_), name in zip(axs.ravel(), [(R_a, t_unit), (R_a, -t_unit), (R_b, t_unit), (R_b, -t_unit)], names):
    P1 = KS @ np.hstack([np.eye(3), np.zeros((3, 1))])
    P2 = KS @ np.hstack([R_, t_])
    Xh = cv2.triangulatePoints(P1, P2, a.T, b.T)
    Xr = (Xh[:3] / Xh[3]).T                                # in camera 1's frame, |t| = 1
    front = (Xr[:, 2] > 0) & ((Xr @ R_.T + t_.ravel())[:, 2] > 0)
    in_front.append((front.sum(), R_, t_))
    ax.scatter(Xr[front, 0], Xr[front, 2], s=16, c='C2', label='in front of both cameras')
    ax.scatter(Xr[~front, 0], Xr[~front, 2], s=16, c='C3', label='behind a camera')
    draw_camera(ax, np.zeros(3), np.eye(3), 'C')
    draw_camera(ax, -R_.T @ t_.ravel(), R_, "C′", 'C0')
    ax.set_title(f'{name}: {front.sum()} of {len(front)} in front', fontsize=19,
                 color='C2' if front.all() else 'k', fontweight='bold' if front.all() else 'normal')
    ax.set_aspect('equal')
    ax.tick_params(labelsize=13)
for ax in axs.ravel():
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-2.2, 3.8)
axs[0, 0].legend(loc='lower left', fontsize=13)
for ax in axs[1]:
    ax.set_xlabel('x (camera 1)', fontsize=15)
for ax in axs[:, 0]:
    ax.set_ylabel('z: depth, |t| = 1', fontsize=15)
fig.tight_layout()
savefig('mvg_four_candidates')
plt.show()
_, R_best, t_best = max(in_front, key=lambda c: c[0])
angle = np.degrees(np.arccos(np.clip((np.trace(R_best @ R_true.T) - 1) / 2, -1, 1)))
cosang = t_best.ravel() @ t_true / np.linalg.norm(t_true)
print(f'chosen candidate: rotation error {angle:.2f} deg, translation direction error {np.degrees(np.arccos(np.clip(cosang, -1, 1))):.2f} deg; '
      f'the length of t ({np.linalg.norm(t_true):.1f} m) cannot be recovered')

## Stereo

Two cameras side by side, looking the same way: the epipolar lines are image rows and a match is a horizontal shift, the **disparity** $d = x_\text{left} - x_\text{right}$. Depth follows from similar triangles: $Z = f b / d$ with the baseline $b$.

First a random-dot stereogram (Julesz, 1960): each image alone is noise, but matching finds a square that is closer than the rest.

In [ ]:
# A random-dot stereogram (Julesz, 1960): each image alone is noise. In the right image a square in the middle
# is shifted by 8 px, the rest by 2 px (disparity d: right(x) = left(x + d)). Matching alone recovers the square.
rng = np.random.default_rng(0)
size, margin = 240, 16
dots = (rng.random((size, size + margin)) > 0.5).astype(np.uint8) * 255
d = np.full((size, size), 2)
d[70:170, 70:170] = 8                                     # the square (in right-image coordinates)
rows, cols = np.mgrid[0:size, 0:size]
left = dots[:, :size].copy()
right = dots[rows, cols + d]
matcher = cv2.StereoSGBM_create(minDisparity=0, numDisparities=16, blockSize=7)
disp = matcher.compute(left, right).astype(np.float32) / 16
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, im, title in zip(axs, [left, right], ['left', 'right']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
axs[2].imshow(np.where(disp >= 0, disp, np.nan), cmap='magma', vmin=0, vmax=10)
axs[2].set_title('disparity found by matching (px)')
axs[2].axis('off')
savefig('mvg_random_dots')
plt.show()
print(f'disparity of the square: {np.median(disp[100:140, 100:140]):.1f} px, of the background: {np.median(disp[10:50, 30:60]):.1f} px')
if SAVE_FIGURES:
    disp_rgb = (plt.get_cmap('magma')(np.clip(np.where(disp >= 0, disp, 0) / 10, 0, 1))[..., :3] * 255).astype(np.uint8)
    save_panels([np.repeat(left[..., None], 3, 2), np.repeat(right[..., None], 3, 2), disp_rgb], 'mvg_random_dots')

### Rectification

Real stereo cameras are never exactly parallel. Rectification rotates both cameras virtually (a homography per image, from `cv2.stereoRectify`) so that the epipolar lines become the same image rows in both images.

In [ ]:
# Rectification: rotate both cameras (virtually, a homography per image) so that the epipolar lines become the
# image rows. Two cameras 1 m apart, each turned inwards by a few degrees and at slightly different heights.
camL = camera([0.5, 1.6, -10], [-1.5, 2.6, 30])                   # world +x is to the left
camR = camera([-0.5, 1.8, -10], [1.2, 2.2, 30])
(imgL, ZL), imgR = photo(street, KS, *camL, WS, HS, depth=True), photo(street, KS, *camR, WS, HS)
R_lr, t_lr = relative_pose(*camL, *camR)
R1r, R2r, P1r, P2r, Q, _, _ = cv2.stereoRectify(KS, None, KS, None, (WS, HS), R_lr, t_lr, alpha=0)
mapL = cv2.initUndistortRectifyMap(KS, None, R1r, P1r, (WS, HS), cv2.CV_32FC1)
mapR = cv2.initUndistortRectifyMap(KS, None, R2r, P2r, (WS, HS), cv2.CV_32FC1)
rectL, rectR = cv2.remap(imgL, *mapL, cv2.INTER_LINEAR), cv2.remap(imgR, *mapR, cv2.INTER_LINEAR)

F_lr = fundamental(KS, *camL, KS, *camR)
points = np.array([(330.0, 90), (130, 160), (300, 240), (235, 290)])    # in the left image
before_L, before_R = imgL.copy(), imgR.copy()
for i, p in enumerate(points):                            # the points and their epipolar lines in the right image
    draw_point(before_L, p, COLOURS[i])
    draw_line(before_R, F_lr @ [p[0], p[1], 1], COLOURS[i])
# where the points are after rectification: the same row in both images is their epipolar line
rect_points = cv2.undistortPoints(points[:, None], KS, None, R=R1r, P=P1r)[:, 0]
matches = project(np.array([backproject(p, ZL[int(p[1]), int(p[0])], KS, *camL) for p in points]), KS, *camR)   # true matches
rect_matches = cv2.undistortPoints(matches[:, None], KS, None, R=R2r, P=P2r)[:, 0]
print('row of each point and of its match after rectification:', np.round(np.c_[rect_points[:, 1], rect_matches[:, 1]], 2).tolist())
after_L, after_R = rectL.copy(), rectR.copy()
for i, p in enumerate(rect_points):
    y = int(round(p[1]))
    cv2.line(after_L, (0, y), (WS, y), COLOURS[i], 2, cv2.LINE_AA)
    cv2.line(after_R, (0, y), (WS, y), COLOURS[i], 2, cv2.LINE_AA)
    draw_point(after_L, p, COLOURS[i])
    draw_point(after_R, rect_matches[i], COLOURS[i])
fig, axs = plt.subplots(2, 2, figsize=(13, 10))
for ax, im, title in zip(axs.ravel(), [before_L, before_R, after_L, after_R],
                         ['left', 'right: epipolar lines of the left points', 'left, rectified', 'right, rectified: epipolar lines = rows']):
    ax.imshow(im)
    ax.set_title(title)
    ax.axis('off')
savefig('mvg_rectification')
plt.show()
save_panels([before_L, before_R, after_L, after_R], 'mvg_rectification')

### Disparity and depth

On a rectified pair, semi-global matching (OpenCV's `StereoSGBM`) searches along each row. We compare with the true disparity $f b / Z$ from the rendered depth.

In [ ]:
# Stereo matching on a rectified pair (parallel cameras, baseline 0.6 m): for every pixel of the left image,
# search along the same row of the right image. Disparity d = x_left - x_right, depth Z = f b / d.
base = 0.6
camL = camera([base / 2, 2.0, -6], [base / 2, 4.0, 30])           # world +x is to the left
camR = camera([-base / 2, 2.0, -6], [-base / 2, 4.0, 30])
(imgL, ZL), imgR = photo(street, KS, *camL, WS, HS, depth=True), photo(street, KS, *camR, WS, HS)
d_true = KS[0, 0] * base / ZL                              # sky: Z = inf, d = 0
sgbm = cv2.StereoSGBM_create(minDisparity=0, numDisparities=64, blockSize=5, P1=8 * 3 * 25, P2=32 * 3 * 25,
                             uniquenessRatio=10, speckleWindowSize=100, speckleRange=2, mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY)
d_est = sgbm.compute(imgL, imgR).astype(np.float32) / 16   # OpenCV returns 16 x the disparity
valid = (d_est > 0) & (d_true > 0.5)
err = np.abs(d_est - d_true)
print(f'disparity: {valid.mean():.0%} of pixels matched; median error {np.median(err[valid]):.2f} px, '
      f'{(err[valid] > 1).mean():.1%} off by more than 1 px')
Z_est = KS[0, 0] * base / np.maximum(d_est, 1e-3)
rel = np.abs(Z_est - ZL) / ZL
print(f'depth: median relative error {np.median(rel[valid]):.1%}')

vmax = np.percentile(d_true[d_true > 0], 99)
fig, axs = plt.subplots(1, 4, figsize=(24, 4.8))
axs[0].imshow(imgL)
axs[0].set_title('left image')
axs[1].imshow(np.where(d_est > 0, d_est, np.nan), cmap='magma', vmin=0, vmax=vmax)
axs[1].set_title('disparity, OpenCV SGBM (px)')
axs[2].imshow(d_true, cmap='magma', vmin=0, vmax=vmax)
axs[2].set_title('true disparity f b / Z (px)')
axs[3].imshow(np.where(valid, err, np.nan), cmap='viridis', vmin=0, vmax=3)
axs[3].set_title('|error| (px)')
for ax in axs:
    ax.axis('off')
savefig('mvg_disparity')
plt.show()
if SAVE_FIGURES:                                         # the panels as separate images for the slide
    cmap = plt.get_cmap('magma')
    to_rgb = lambda d: (cmap(np.clip(np.nan_to_num(d, nan=0) / vmax, 0, 1))[..., :3] * 255).astype(np.uint8)
    est = to_rgb(np.where(d_est > 0, d_est, 0))
    est[d_est <= 0] = 255                                  # unmatched: white
    errimg = (plt.get_cmap('viridis')(np.clip(err / 3, 0, 1))[..., :3] * 255).astype(np.uint8)
    errimg[~valid] = 255
    save_panels([imgL, imgR, est, to_rgb(d_true), errimg], 'mvg_disparity')

### How accurate is depth from two views?

A disparity error of $\sigma$ pixels changes the depth by about $\Delta Z \approx \frac{Z^2}{f b} \sigma$: the error grows with the *square* of the distance and shrinks with the baseline.

In [ ]:
# How accurate is triangulated depth? Two parallel cameras, side by side (baseline b), known K, R, t.
# Exact matches + 0.5 px noise, linear triangulation; compare with the true depth.
sigma = 0.5
fig, ax = plt.subplots(figsize=(7, 4.5))
for i, base in enumerate([0.25, 1.0, 4.0]):
    camL = camera([base / 2, 1.6, -10], [base / 2, 1.6, 30])       # world +x is to the left
    camR = camera([-base / 2, 1.6, -10], [-base / 2, 1.6, 30])
    pL, pR, X = dense_matches(camL, camR, n=6000, seed=i, max_depth=1e9)
    rng = np.random.default_rng(i)
    PL, PR = KS @ np.hstack([camL[0], camL[1][:, None]]), KS @ np.hstack([camR[0], camR[1][:, None]])
    Xh = cv2.triangulatePoints(PL, PR, (pL + rng.normal(0, sigma, pL.shape)).T, (pR + rng.normal(0, sigma, pR.shape)).T)
    Xe = (Xh[:3] / Xh[3]).T
    Z_true = (X @ camL[0].T + camL[1])[:, 2]
    Z_est = (Xe @ camL[0].T + camL[1])[:, 2]
    err = np.abs(Z_est - Z_true)
    ax.scatter(Z_true, np.maximum(err, 1e-4), s=2, color=f'C{i}', alpha=0.3)
    Zs = np.geomspace(3, 300, 50)
    ax.plot(Zs, Zs ** 2 * sigma / (KS[0, 0] * base), color=f'C{i}', lw=2, label=f'baseline {base:g} m')
    print(f'baseline {base:5.2f} m: median depth error at 20-40 m: {np.median(err[(Z_true > 20) & (Z_true < 40)]):.2f} m')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(3, 300)
ax.set_ylim(1e-3, 1e3)
ax.set_xlabel('true depth Z (m)')
ax.set_ylabel('depth error (m)')
ax.set_title(f'Triangulated depth, {sigma} px noise; lines: Z² σ / (f b)')
ax.legend()
ax.grid(alpha=0.3, which='both')
savefig('mvg_depth_error')
plt.show()

## Structure from motion

Many photos, no known cameras. Incremental SfM: start from two views ($E$, then $R$ and $\mathbf{t}$, then triangulate), add one camera at a time from 2D-3D matches to the points already reconstructed (Perspective-n-Point, PnP), triangulate the points it adds, and finally refine all cameras and points together: **bundle adjustment** minimises the total reprojection error.

The correspondences come from the renderer (each sampled surface point is tracked through every photo that sees it), with 0.5 px of noise; everything after matching is what a real pipeline does. The reconstruction is only defined up to a similarity (rotation, translation, scale), so for the plots we align it to the true scene.

In [ ]:
# Incremental structure from motion on the street: 7 photos taken while walking down the street, each looking
# obliquely at the buildings on the left. Correspondences come from the renderer (every 3D surface point we sample is tracked
# through all photos that see it), plus 0.5 px of noise: the steps after matching are the real ones.
from scipy.optimize import least_squares
from scipy.sparse import lil_matrix

N_CAM = 7
sfm_cams = [camera([-1.0, 1.6, -6 + 3.0 * i], [20.0, 4, 14 + 3.0 * i]) for i in range(N_CAM)]   # world +x is to the left
sfm_renders = [photo(street, KS, *cam, WS, HS, depth=True, ss=1) for cam in sfm_cams]

# tracks: 3D points on visible surfaces (closer than 45 m), observed in every camera that sees them
rng = np.random.default_rng(0)
pts_true, cols = [], []
for (img, Z), cam in zip(sfm_renders, sfm_cams):
    p = np.stack([rng.uniform(0, WS, 500), rng.uniform(0, HS, 500)], 1)
    z = Z[p[:, 1].astype(int), p[:, 0].astype(int)]
    ok = z < 45
    pts_true += [backproject(q, d, KS, *cam) for q, d in zip(p[ok], z[ok])]
    cols += list(img[p[ok, 1].astype(int), p[ok, 0].astype(int)] / 255)
pts_true, cols = np.array(pts_true), np.array(cols)
obs = {}                                                 # (camera, point) -> observed pixel
for c, ((img, Z), cam) in enumerate(zip(sfm_renders, sfm_cams)):
    q = project(pts_true, KS, *cam)
    zc = (pts_true @ cam[0].T + cam[1])[:, 2]
    inside = (zc > 0) & (q[:, 0] >= 0) & (q[:, 0] < WS) & (q[:, 1] >= 0) & (q[:, 1] < HS)
    for j in np.flatnonzero(inside):
        if abs(Z[int(q[j, 1]), int(q[j, 0])] - zc[j]) < 0.01 * zc[j]:
            obs[c, j] = q[j] + rng.normal(0, 0.5, 2)
print(f'{len(pts_true)} points, {len(obs)} observations in {N_CAM} photos')

def seen(c):
    return {j for (c_, j) in obs if c_ == c}

def triangulate_new(c1, c2, poses, X):
    P1 = KS @ np.hstack([poses[c1][0], poses[c1][1][:, None]])
    P2 = KS @ np.hstack([poses[c2][0], poses[c2][1][:, None]])
    C1, C2 = [-poses[c][0].T @ poses[c][1] for c in (c1, c2)]
    for j in sorted((seen(c1) & seen(c2)) - set(X)):
        Xh = cv2.triangulatePoints(P1, P2, obs[c1, j][:, None], obs[c2, j][:, None])[:, 0]
        Y = Xh[:3] / Xh[3]
        # keep only well-conditioned points: in front of both cameras, rays at least 2 degrees apart, small reprojection error
        r1, r2 = Y - C1, Y - C2
        angle = np.degrees(np.arccos(np.clip(r1 @ r2 / np.linalg.norm(r1) / np.linalg.norm(r2), -1, 1)))
        err = max(np.linalg.norm(project(Y[None], KS, *poses[c])[0] - obs[c, j]) for c in (c1, c2))
        front = all((poses[c][0] @ Y + poses[c][1])[2] > 0 for c in (c1, c2))
        if front and angle > 2 and err < 3:
            X[j] = Y

# 1) two views: E with RANSAC, R and t (|t| = 1), triangulate
common = sorted(seen(0) & seen(1))
a_, b_ = np.array([obs[0, j] for j in common]), np.array([obs[1, j] for j in common])
E01, _ = cv2.findEssentialMat(a_, b_, KS, cv2.RANSAC, 0.999, 1.0)
_, R01, t01, _ = cv2.recoverPose(E01, a_, b_, KS)
poses = {0: (np.eye(3), np.zeros(3)), 1: (R01, t01.ravel())}
X_rec = {}
triangulate_new(0, 1, poses, X_rec)
stages = [(dict(poses), dict(X_rec), None)]
# 2) one camera at a time: its pose from 2D-3D matches (PnP), then triangulate the points it adds
for c in range(2, N_CAM):
    known = sorted(seen(c) & set(X_rec))
    Xk, xk = np.array([X_rec[j] for j in known]), np.array([obs[c, j] for j in known])
    ok, rvec, tvec, inl = cv2.solvePnPRansac(Xk, xk, KS, None, reprojectionError=3.0, iterationsCount=500)
    ok, rvec, tvec = cv2.solvePnP(Xk[inl[:, 0]], xk[inl[:, 0]], KS, None, rvec, tvec, useExtrinsicGuess=True)   # refine on the inliers
    poses[c] = (cv2.Rodrigues(rvec)[0], tvec.ravel())
    stages.append((dict(poses), dict(X_rec), c))          # new camera, before its points
    triangulate_new(c - 1, c, poses, X_rec)
stages.append((dict(poses), dict(X_rec), None))

# 3) bundle adjustment: all poses (camera 0 fixed) and all points, minimising the reprojection error
ids = sorted(X_rec)
index = {j: i for i, j in enumerate(ids)}
pairs_ = [(c, j) for (c, j) in obs if j in index and c in poses]
def pack(poses, X):
    return np.concatenate([np.r_[cv2.Rodrigues(poses[c][0])[0].ravel(), poses[c][1]] for c in range(1, N_CAM)] + [np.array([X[j] for j in ids]).ravel()])
def unpack(v):
    ps = {0: poses[0]}
    for c in range(1, N_CAM):
        r = v[6 * (c - 1): 6 * c]
        ps[c] = (cv2.Rodrigues(r[:3])[0], r[3:])
    return ps, v[6 * (N_CAM - 1):].reshape(-1, 3)
def residuals(v):
    ps, P = unpack(v)
    out = []
    for c in range(N_CAM):
        js = [index[j] for (c_, j) in pairs_ if c_ == c]
        q = project(P[js], KS, *ps[c])
        out.append((q - np.array([obs[c, ids[i]] for i in js])).ravel())
    return np.concatenate(out)
sparsity = lil_matrix((2 * len(pairs_), 6 * (N_CAM - 1) + 3 * len(ids)), dtype=int)
row = 0
for c in range(N_CAM):
    for (c_, j) in pairs_:
        if c_ != c:
            continue
        if c > 0:
            sparsity[row:row + 2, 6 * (c - 1): 6 * c] = 1
        sparsity[row:row + 2, 6 * (N_CAM - 1) + 3 * index[j]: 6 * (N_CAM - 1) + 3 * index[j] + 3] = 1
        row += 2
v0 = pack(poses, X_rec)
rms_before = np.sqrt(np.mean(residuals(v0) ** 2))
print(f'before bundle adjustment: RMS reprojection error {rms_before:.2f} px')
fit_ = least_squares(residuals, v0, jac_sparsity=sparsity, x_scale='jac', method='trf', loss='huber', f_scale=2.0, max_nfev=300)
rms_after = np.sqrt(np.mean(fit_.fun ** 2))
print(f'after bundle adjustment:  RMS reprojection error {rms_after:.2f} px')
ps_ba, P_ba = unpack(fit_.x)
stages.append((ps_ba, dict(zip(ids, P_ba)), None))

# into the world frame: the reconstruction is only defined up to a similarity (rotation, translation, scale: 7 DoF).
# For plotting and for measuring errors we apply the similarity that best maps its points onto the true ones (Umeyama).
C_true = np.array([-R.T @ t for R, t in sfm_cams])
def similarity(A, B):
    """s, R, t minimising |s R A + t - B|^2 (rows are points)"""
    ma, mb = A.mean(0), B.mean(0)
    U_, S_, Vt_ = np.linalg.svd((B - mb).T @ (A - ma))
    D = np.diag([1, 1, np.sign(np.linalg.det(U_ @ Vt_))])
    R_ = U_ @ D @ Vt_
    s_ = (S_ * np.diag(D)).sum() / ((A - ma) ** 2).sum()
    return s_, R_, mb - s_ * R_ @ ma
def to_world(ps, X):
    js = sorted(X)
    s_, R_, t_ = similarity(np.array([X[j] for j in js]), pts_true[js])
    f = lambda Y: s_ * np.asarray(Y) @ R_.T + t_
    return {c: (f(-R.T @ t), R @ R_.T) for c, (R, t) in ps.items()}, {j: f(Y) for j, Y in X.items()}

def sfm_top(ax, ps, X, new=None, truth=False):
    # from above, the street running left to right (world z), the buildings the cameras look at on top (world +x)
    w, h, d = BLDG
    for x0 in (-X_FACE - w, X_FACE):
        for j in range(4):
            ax.add_patch(plt.Rectangle((15 * j, x0), d, w, fill=False, ec='0.6', ls='--', lw=0.8))
    cams_w, X_w = to_world(ps, X)
    js = list(X_w)
    P = np.array([X_w[j] for j in js])
    high = P[:, 1] > 0.3                                 # points on the walls; the rest lie on the ground (road, pavements)
    ax.scatter(P[~high, 2], P[~high, 0], s=2, c='tan', label='ground points')
    ax.scatter(P[high, 2], P[high, 0], s=4, c='k', label='wall points')
    if truth:
        ax.plot(C_true[:, 2], C_true[:, 0], 'o', mfc='none', mec='C0', ms=12, mew=1.5, label='true camera centres')
    for c, (C, Rw) in cams_w.items():
        colour = 'C3' if c == new else 'k'
        dvec = Rw.T @ np.array([0, 0, 3.0])
        ax.plot(C[2], C[0], 's', color=colour, ms=7)
        ax.annotate('', (C[2] + dvec[2], C[0] + dvec[0]), (C[2], C[0]), arrowprops=dict(arrowstyle='->', color=colour, lw=1.5))
    ax.set_xlim(-9, 37)
    ax.set_ylim(-4.5, 17)
    ax.set_aspect('equal')
    ax.axis('off')

drift = lambda ps, X: np.linalg.norm(np.array([to_world(ps, X)[0][c][0] for c in range(N_CAM)]) - C_true, axis=1)
print('camera centre error (m), incremental:     ', drift(*stages[-2][:2]).round(2))
print('camera centre error (m), after adjustment:', drift(*stages[-1][:2]).round(2))
frames = [(stages[0], 'two views: E -> R, t; triangulate', False),
          (stages[1], 'camera 3: pose from known points (PnP)', False),
          (stages[-2], f'all {N_CAM} cameras: RMS error {rms_before:.1f} px', True),
          (stages[-1], f'bundle adjustment: {rms_after:.2f} px', True)]
fig, axs = plt.subplots(2, 2, figsize=(16, 9))
for i, (ax, ((ps, X, new), title, truth)) in enumerate(zip(axs.ravel(), frames), 1):
    sfm_top(ax, ps, X, new, truth)
    ax.set_title(title, fontsize=13)
    if SAVE_FIGURES:
        f1, a1 = plt.subplots(figsize=(9, 4.8))
        sfm_top(a1, ps, X, new, truth)
        a1.set_title(title, fontsize=15)
        a1.legend(loc='lower right', fontsize=10)
        f1.savefig(f'figures/mvg_sfm_{i}.png', dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(f1)
axs[1, 0].legend(loc='lower right')
plt.show()